# Project 1: Advanced EDA & Feature Engineering Overview

## Project Goal

Transform raw, unrefined e-commerce transaction data into a clean, fully engineered dataset ready for machine learning models.

## 1. Data Cleaning & Imputation

### Missing Value Handling

- Filled **309 missing values** in `CouponCode` with `NO_COUPON` instead of dropping the rows.
- This preserved **100% of the transaction data**.

### Outlier Neutralization

- Applied **Interquartile Range (IQR) boundary capping (Winsorization)** to `TotalPrice`.
- Extreme values above **$3,330.41** were capped at the upper threshold.
- This reduced the influence of extreme values without removing transaction records.

## 2. Feature Engineering & Vectorization

### Effective Item Price

Created an `EffectiveItemPrice` feature using:

`TotalPrice / ItemsInCart`

This captures the average amount spent per item in an order.

### High Value Order Flag

Created a binary `HighValueOrder` indicator:

- `1` = order is within the top 25% revenue bracket
- `0` = order is below the top 25% revenue bracket

### Discount Flag

Created a binary `DiscountFlag` to identify whether a customer used a coupon code:

- `1` = coupon used
- `0` = no coupon used

### One-Hot Encoding

Converted categorical variables into binary features using one-hot encoding:

- `Product`
- `PaymentMethod`
- `OrderStatus`
- `ReferralSource`

This transformed categorical information into a numerical format suitable for machine learning models.

## 3. Quality Assurance & Output

### Collinearity Verification

- Evaluated **Pearson correlation** across numeric features.
- Verified that no pair of numeric features had a correlation above **0.80**.
- This helped reduce the risk of multicollinearity affecting model training.

### Data Contract Export

- Saved the cleaned and transformed dataset as:

`cleaned_analytics_dataset.csv`

- The exported dataset provides a consistent, model-ready input for subsequent machine learning workflows and helps reduce differences between training and serving data.

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

# Load Dataset
file_path = "Dataset for Data Analytics - Sheet1.csv"
df = pd.read_csv(file_path)

# PHASE 1: INPUT FIDELITY

# 1. Missing Value Imputation Decision Matrix
missing_pct = df.isnull().mean()

for col in df.columns:
    pct = missing_pct[col]
    if pct == 0:
        continue
    elif pct < 0.05:
        # Drop rows for minimal missingness (<5%)
        df = df.dropna(subset=[col])
    elif 0.05 <= pct <= 0.20:
        # Statistical Imputation (Median for numeric, Mode for categorical)
        if np.issubdtype(df[col].dtype, np.number):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])
    else:
        # KNN Imputation for high missingness (>20%)
        if np.issubdtype(df[col].dtype, np.number):
            knn_imputer = KNNImputer(n_neighbors=5)
            df[[col]] = knn_imputer.fit_transform(df[[col]])

# 2. Outlier Neutralization (IQR Winsorization)
num_cols = df.select_dtypes(include=[np.number]).columns

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    # Cap boundaries using numpy.clip to maintain sequential length
    df[col] = np.clip(df[col], lower_bound, upper_bound)

# PHASE 2: VECTORIZED ENGINE & FEATURE ENGINEERING

# 1. Feature Engineering (3 New Predictive Metrics)
# Example interaction, ratio, and polynomial features
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_features) >= 2:
    f1, f2 = numeric_features[0], numeric_features[1]
    df['fe_interaction_ratio'] = df[f1] / (df[f2] + 1e-5)
    df['fe_aggregate_sum'] = df[f1] + df[f2]
    df['fe_log_transformed'] = np.log1p(np.abs(df[f1]))

# 2. Categorical Orthogonal Encoding (OHE)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# 3. Collinearity Eradication (Threshold > 0.80)
corr_matrix = df.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Identify collinear pairs > 0.80
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.80)]
df_cleaned = df.drop(columns=to_drop)

# PHASE 3: OUTPUT SERVING

# Export production-ready dataset
df_cleaned.to_csv("cleaned_production_dataset.csv", index=False)
print(f"Pipeline Execution Complete. Cleaned Shape: {df_cleaned.shape}")

Pipeline Execution Complete. Cleaned Shape: (1200, 1925)
